# 3D Augmentation Visualization

This notebook visualizes the 3D augmentation transforms applied during training.
Use it to verify augmentation parameters and inspect augmented volumes.

## Usage
Set `DATA_DIR` to your 3D data directory and run the cells.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from data.augmentation_3d import (
    GaussianNoiseTransform, GaussianBlurTransform,
    BrightnessTransform, GammaTransform,
    RandomRotFlip, MirrorTransform,
)

In [ ]:
# --- Configure paths ---
DATA_DIR = 'data/3d'  # Update to your data directory

img_file = f'{DATA_DIR}/imagesTr/crc_007.nii.gz'
gt_file  = f'{DATA_DIR}/labelsTr/crc_007.nii.gz'

In [ ]:
def draw_volume_slices(volume, title='Volume'):
    """Display 3 orthogonal mid-slices of a 3D volume."""
    z_n = volume.shape[0] // 2
    y_n = volume.shape[1] // 2
    x_n = volume.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(volume[z_n, :, :], cmap='gray')
    axes[0].set_title(f'Axial (z={z_n})')
    axes[0].axis('off')

    axes[1].imshow(volume[:, y_n, :], cmap='gray')
    axes[1].set_title(f'Coronal (y={y_n})')
    axes[1].axis('off')

    axes[2].imshow(volume[:, :, x_n], cmap='gray')
    axes[2].set_title(f'Sagittal (x={x_n})')
    axes[2].axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Load original volume
img_sitk = sitk.ReadImage(img_file)
gt_sitk  = sitk.ReadImage(gt_file)

img_np = sitk.GetArrayFromImage(img_sitk).astype(np.float32)
gt_np  = sitk.GetArrayFromImage(gt_sitk)

# Z-score normalization (matching training preprocessing)
img_np = (img_np - img_np.mean()) / (img_np.std() + 1e-8)

print(f'Volume shape: {img_np.shape}')
print(f'Spacing: {img_sitk.GetSpacing()}')
draw_volume_slices(img_np, title='Original Volume')

## Apply Individual Transforms

In [ ]:
# Prepare sample in the format expected by transforms
sample = {
    'image': np.expand_dims(img_np, axis=0),  # [1, Z, Y, X]
    'label': np.expand_dims(gt_np, axis=0),
}

In [ ]:
# Gaussian Noise
aug = GaussianNoiseTransform(noise_variance=(0, 0.07), p_per_sample=1.0)
result = aug(sample.copy())
draw_volume_slices(result['image'][0], title='Gaussian Noise (var=0~0.07)')

In [ ]:
# Gaussian Blur
aug = GaussianBlurTransform(blur_sigma=(1, 2.5), different_sigma_per_channel=False,
                            p_per_channel=0.25, p_per_sample=1.0)
result = aug(sample.copy())
draw_volume_slices(result['image'][0], title='Gaussian Blur (sigma=1~2.5)')

In [ ]:
# Brightness
aug = BrightnessTransform(mu=0, sigma=1, per_channel=False,
                          p_per_channel=0.25, p_per_sample=1.0)
result = aug(sample.copy())
draw_volume_slices(result['image'][0], title='Brightness (mu=0, sigma=1)')

In [ ]:
# Gamma Correction
aug = GammaTransform(gamma_range=(0.25, 1), per_channel=False, p_per_sample=1.0)
result = aug(sample.copy())
draw_volume_slices(result['image'][0], title='Gamma Correction (0.25~1)')

## Apply Full Training Pipeline

In [ ]:
full_pipeline = transforms.Compose([
    GaussianNoiseTransform(noise_variance=(0, 0.07), p_per_sample=0.25),
    GaussianBlurTransform(blur_sigma=(1, 2.5), different_sigma_per_channel=False,
                          p_per_channel=0.25, p_per_sample=0.25),
    BrightnessTransform(mu=0, sigma=1, per_channel=False,
                        p_per_channel=0.25, p_per_sample=0.25),
    GammaTransform(gamma_range=(0.25, 1), per_channel=False, p_per_sample=0.25),
    RandomRotFlip(p_per_sample=0.25),
    MirrorTransform(p_per_sample=0.25),
])

# Apply multiple times to see variation
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
z_n = img_np.shape[0] // 2

for i in range(9):
    result = full_pipeline(sample.copy())
    ax = axes[i // 3][i % 3]
    ax.imshow(result['image'][0][z_n], cmap='gray')
    ax.set_title(f'Augmentation #{i+1}')
    ax.axis('off')

plt.suptitle('9 Random Augmentations (Axial mid-slice)')
plt.tight_layout()
plt.show()